# Arm prompt probe launcher (`libero_10` task 0)

Runs `validation/arm_prompt_probe.py` to isolate whether a `grab` step fails from:
- prompt wording mismatch
- per-skill step budget (`MAX_SKILL_STEPS`)
- or both.

Each prompt is scored on the grab subtask (`check_subtask`), not full-task
success. The first prompt is the `task_description` control.

## 1 - Clone repo

In [ ]:
import getpass

github_token = getpass.getpass('GitHub PAT (repo scope): ')

In [ ]:
import os

if os.path.exists('/content/Longor'):
    os.chdir('/content/Longor')
    print('pulling repo...')
    !git pull
else:
    os.environ['CLONE_URL'] = f"https://{github_token}@github.com/pmish11/Longor-Robotics.git"
    !git clone $CLONE_URL "/content/Longor"
    os.chdir('/content/Longor')

## 2 - Headless rendering + app deps

In [ ]:
import os

os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MUJOCO_GL'] = 'egl'

!apt-get update
!apt-get install -y cmake libegl-dev libgl1-mesa-dev libgles2-mesa-dev
%pip install egl_probe hf-egl-probe --no-build-isolation
%pip install -U pip
%pip install -r deps/requirements-libero.txt
%pip install -e .

## 3 - Pi0.5 (OpenPI) venv

In [ ]:
import os, subprocess, sys

VLA_PYTHON = '/content/Longor/.vla-venv/bin/python'
OPENPI_DIR = '/content/openpi'

def vla_pip(*args):
    subprocess.run([VLA_PYTHON, '-m', 'pip', *args], check=True)

%pip install --quiet virtualenv
if not os.path.exists('/content/Longor/.vla-venv'):
    subprocess.run([sys.executable, '-m', 'virtualenv', '/content/Longor/.vla-venv'], check=True)
vla_pip('install', '-U', 'pip')

if not os.path.isdir(OPENPI_DIR):
    subprocess.run([
        'git', 'clone', '--recurse-submodules',
        'https://github.com/Physical-Intelligence/openpi', OPENPI_DIR,
    ], check=True)
vla_pip('install', '-e', OPENPI_DIR)
vla_pip('install', '-r', 'deps/requirements-vla-pi05-linux.txt')
vla_pip('install', '-e', '.')

## 4 - Start VLA server + health

In [ ]:
import os, subprocess, time, urllib.request

with open('.env', 'w', encoding='utf-8') as f:
    f.write('LOG_LEVEL=INFO\nSAVE_VIDEO=0\n')

VLA_LOG = '/tmp/arm_prompt_probe_vla_server.log'
vla_python = '/content/Longor/.vla-venv/bin/python'
log = open(VLA_LOG, 'w', buffering=1)
proc = subprocess.Popen(
    [vla_python, 'services/run_vla_server.py'],
    cwd='/content/Longor',
    stdout=log,
    stderr=subprocess.STDOUT,
    env=dict(os.environ),
)

health_ok = False
for elapsed in range(0, 240, 5):
    try:
        resp = urllib.request.urlopen('http://localhost:8765/health', timeout=5)
        print('Health:', resp.read().decode())
        health_ok = True
        break
    except Exception:
        print(f'Waiting for /health ({elapsed}s)...')
        time.sleep(5)

if not health_ok:
    raise RuntimeError(f'VLA server not healthy; see {VLA_LOG}')

## 5 - Run prompt sweep

In [ ]:
!python validation/arm_prompt_probe.py --suite libero_10 --task 0 --init-state-idx 0 --object "alphabet soup" --skill-step-budgets 120 280 --vla-actions-per-chunk 5